# 04A Campus data through MCP

**Problem:** Students want to find affordable events for their department and save a shortlist. Build a real MCP connection that discovers a structured database tool, queries fictional events, reads a rules resource and retrieves a planning prompt.

**Core build: about 40 minutes using the supplied starter.** Run the working example, then change the catalogue and client query and add a new tool. A CPU runtime is enough. This notebook calls a real MCP server through an in-memory MCP client. It does not call an LLM or make a real event registration.

**First success:** CSE events costing at most 100 INR are E101, E102, E104 and E105. Free CSE/ALL events are E101 and E104.

**Reference:** Inspired by [AI Engineering Hub / LlamaIndex MCP](https://github.com/patchy631/ai-engineering-hub/tree/042aeb5067384981eeba6a07fa882f436bb1660d/llamaindex-mcp). This is an independently written classroom adaptation: FastMCP and SQLite replace the original model/client stack; structured parameters replace arbitrary SQL strings. The original project remains the reference for a larger application.


## 1 Prepare the MCP packages

Run in a fresh Colab CPU runtime. This cell reuses matching versions or installs only the pinned MCP packages. If Colab requests a restart, restart and run from here again.

In [ ]:
# Install only the two MCP packages used by this lab, if needed.
import importlib.metadata as metadata
import subprocess, sys
required = {"fastmcp": "2.14.5", "mcp": "1.30.0"}
missing = []
for package, version in required.items():
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        missing.append(f"{package}=={version}")
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Setup ready:", {name: metadata.version(name) for name in required})
print("CPU lab: no model account, API key, GPU or public server is needed.")


## 2 Create the fictional event catalogue

Inspect EVENT_ROWS. Running this cell with RESET_DEMO=True resets only this dedicated demo database and its shortlist. Use False to keep existing demo records.

In [ ]:
# This is a fictional catalogue for classroom practice.
import json, sqlite3
from pathlib import Path
from fastmcp import FastMCP, Client

# Use only a project-local database in this notebook runtime.
WORK_DIR = Path.cwd() / "bgs_day2_mcp"
WORK_DIR.mkdir(exist_ok=True)
DB_PATH = WORK_DIR / "campus_events.sqlite3"

# Rerunning this cell with True rebuilds this demo catalogue and clears its shortlist.
# Change to False if you want to preserve an existing demo database instead.
RESET_DEMO = True
EVENT_ROWS = [
    ("E101", "Intro to AI", "CSE", "2026-11-10", 0),
    ("E102", "Python Hack Night", "CSE", "2026-11-11", 100),
    ("E103", "Robotics Lab", "ECE", "2026-11-12", 150),
    ("E104", "Design a Better Campus", "ALL", "2026-11-13", 0),
    ("E105", "Secure AI Challenge", "CSE", "2026-11-14", 50),
]
with sqlite3.connect(DB_PATH) as database:
    database.execute("CREATE TABLE IF NOT EXISTS events (event_id TEXT PRIMARY KEY, title TEXT, department TEXT, event_date TEXT, fee_inr INTEGER)")
    database.execute("CREATE TABLE IF NOT EXISTS shortlist (event_id TEXT PRIMARY KEY)")
    if RESET_DEMO:
        database.execute("DELETE FROM shortlist")
        database.execute("DELETE FROM events")
    # Fixed placeholders keep the values separate from SQL instructions.
    database.executemany("INSERT OR IGNORE INTO events VALUES (?, ?, ?, ?, ?)", EVENT_ROWS)
print("Fictional events available:", len(EVENT_ROWS))
print("Demo database:", DB_PATH.name)


## 3 Expose tools context and a prompt

Inspect the parameterised query and the tool return values. A resource returns context; a prompt returns a template. Neither automatically runs an action or a model.

In [ ]:
# All database operations below are invoked through MCP tool calls.
def build_campus_server(database_path):
    server = FastMCP("BGS Campus Events")

    def read_rows(sql, parameters=()):
        # Each tool call opens its own connection; no connection crosses threads.
        with sqlite3.connect(database_path) as database:
            database.row_factory = sqlite3.Row
            return [dict(row) for row in database.execute(sql, parameters).fetchall()]

    @server.tool
    def list_events(department: str = "", max_fee: int = 100) -> dict:
        """Find fictional events by department and maximum fee in INR. ALL events are open to every department."""
        department = department.strip().upper()
        if department not in {"", "CSE", "ECE", "MECH", "ALL"}:
            return {"ok": False, "error": "UNKNOWN_DEPARTMENT", "events": []}
        if max_fee < 0 or max_fee > 5000:
            return {"ok": False, "error": "USE_FEE_0_TO_5000", "events": []}
        # The client supplies values, never an arbitrary SQL query.
        rows = read_rows(
            "SELECT * FROM events WHERE (? = '' OR department = ? OR department = 'ALL') AND fee_inr <= ? ORDER BY event_date, event_id",
            (department, department, max_fee))
        return {"ok": True, "events": rows, "count": len(rows), "simulation": True}

    @server.tool
    def get_event(event_id: str) -> dict:
        """Read one fictional event by its exact ID."""
        rows = read_rows("SELECT * FROM events WHERE event_id = ?", (event_id,))
        return {"ok": bool(rows), "event": rows[0] if rows else None,
                "error": None if rows else "EVENT_NOT_FOUND", "simulation": True}

    @server.tool
    def save_shortlist(event_id: str) -> dict:
        """Save a known event to this local shortlist. This is not registration or payment."""
        rows = read_rows("SELECT event_id FROM events WHERE event_id = ?", (event_id,))
        if not rows:
            return {"ok": False, "error": "EVENT_NOT_FOUND", "simulation": True}
        with sqlite3.connect(database_path) as database:
            database.execute("INSERT OR IGNORE INTO shortlist VALUES (?)", (event_id,))
        return {"ok": True, "saved_event_id": event_id, "simulation": True}

    @server.tool
    def read_shortlist() -> dict:
        """Read events saved in the local shortlist."""
        rows = read_rows("SELECT events.* FROM events JOIN shortlist USING (event_id) ORDER BY event_id")
        return {"ok": True, "events": rows, "count": len(rows), "simulation": True}

    @server.resource("campus://event-rules", mime_type="application/json")
    def event_rules() -> str:
        return json.dumps({"data": "fictional classroom events", "currency": "INR",
            "ALL_department": "open to every department", "shortlist": "local save only; no registration or payment"}, indent=2)

    @server.prompt
    def plan_campus_visit(department: str, budget_inr: str) -> str:
        """Prepare a template for planning from returned event data; no model is called."""
        return (f"Use only returned event facts to suggest a visit for {department} with a budget of {budget_inr} INR. "
                "Check dates and explain that these are fictional practice events.")

    return server

server = build_campus_server(DB_PATH)
campus_client = Client(server)
print("A real MCP server and in-memory MCP client are ready.")


## 4 Discover through a real MCP client

Run client discovery and read the resource and prompt. These are actual protocol operations using an in-memory transport; calling Python functions directly would not demonstrate MCP.

In [ ]:
# An in-memory transport still performs the MCP client/server protocol.
async with campus_client:
    tools = await campus_client.list_tools()
    resources = await campus_client.list_resources()
    prompts = await campus_client.list_prompts()
    rules = await campus_client.read_resource("campus://event-rules")
    template = await campus_client.get_prompt("plan_campus_visit", {"department": "CSE", "budget_inr": "100"})
print("Tools:", [tool.name for tool in tools])
print("Resource:", str(resources[0].uri))
print(rules[0].text)
print("Prompt template:", template.messages[0].content.text)
assert {"list_events", "get_event", "save_shortlist", "read_shortlist"}.issubset({tool.name for tool in tools})
assert any(prompt.name == "plan_campus_visit" for prompt in prompts)


## 5 Query the server and check its results

Follow the request arguments to the returned event data. The checks calculate expected IDs directly from EVENT_ROWS in Python, then compare them with the actual MCP replies. They follow your catalogue edits instead of requiring the original event list. Run the data cell with RESET_DEMO=True so EVENT_ROWS describes the complete catalogue for these checks. An unknown ID returns an explicit missing result.

In [ ]:
# These are client arguments; we do not directly call a Python server function.
async with campus_client:
    cse_result = await campus_client.call_tool("list_events", {"department": "CSE", "max_fee": 100})
    free_result = await campus_client.call_tool("list_events", {"department": "CSE", "max_fee": 0})
    missing_result = await campus_client.call_tool("get_event", {"event_id": "DOES-NOT-EXIST"})
events = cse_result.data
print(json.dumps(events, indent=2))
print("Free CSE/ALL events:", [row["event_id"] for row in free_result.data["events"]])
print("Unknown event:", missing_result.data)
# Calculate expectations from the edited fixture in Python, independently of SQL.
# Rebuild the catalogue with RESET_DEMO=True so EVENT_ROWS describes every record.
ordered_rows = sorted(EVENT_ROWS, key=lambda row: (row[3], row[0]))
expected_cse_ids = [row[0] for row in ordered_rows
                    if row[2] in {"CSE", "ALL"} and row[4] <= 100]
expected_free_ids = [row[0] for row in ordered_rows
                     if row[2] in {"CSE", "ALL"} and row[4] <= 0]
assert [row["event_id"] for row in events["events"]] == expected_cse_ids
assert [row["event_id"] for row in free_result.data["events"]] == expected_free_ids
assert missing_result.data["ok"] is False and missing_result.data["event"] is None


## 6 Choose whether to save a shortlist

The write operation stays off by default. Change CONFIRM_SAVE yourself to permit this local save. This is a client-side choice, not a server guarantee that every caller obtained human permission.

In [ ]:
# The client decides whether to call the write tool.
# Set True yourself only if you want to save this fictional event locally.
CONFIRM_SAVE = False
saved_reply = None
if CONFIRM_SAVE:
    async with campus_client:
        saved_reply = (await campus_client.call_tool("save_shortlist", {"event_id": "E101"})).data
    print("Actual save result:", saved_reply)
else:
    print("Save not confirmed: the write tool was not called.")
async with campus_client:
    shortlist = (await campus_client.call_tool("read_shortlist", {})).data
print("Actual local shortlist:", shortlist)
if CONFIRM_SAVE:
    assert saved_reply["ok"] is True
    assert any(row["event_id"] == "E101" for row in shortlist["events"])


## 7 Your first build tasks

Task 1: change the department and budget below and predict the result. Task 2: add a fictional event with a new event ID to EVENT_ROWS, rebuild the catalogue with RESET_DEMO=True, then rerun the server and query cells and show its effect. The query checks calculate their expectations from your edited catalogue. Explain one SQL placeholder.

In [ ]:
# STUDENT TASK 1: change both values, run, then explain the returned events.
YOUR_DEPARTMENT = "ECE"
YOUR_MAX_FEE = 150
async with campus_client:
    your_result = (await campus_client.call_tool("list_events", {
        "department": YOUR_DEPARTMENT, "max_fee": YOUR_MAX_FEE})).data
print(json.dumps(your_result, indent=2))
if your_result["ok"]:
    assert all(row["fee_inr"] <= YOUR_MAX_FEE for row in your_result["events"])
print("Now add a fictional event with a new ID to EVENT_ROWS. Rebuild with RESET_DEMO=True, then rerun the server and query cells.")


## 8 Add your own MCP capability

Task 3: implement events_on_date as a newly registered MCP tool. Use a fixed query and a date parameter. Rediscover it and call it through the client. The starter does not mark this task completed for you.

In [ ]:
# STUDENT TASK 3: add an @server.tool function named events_on_date(event_date).
# Hint: query WHERE event_date = ? with a parameter, then return {"ok": True, "events": rows}.
# Place the function below, run this cell, and discover it through the client again.

async with campus_client:
    refreshed_tools = await campus_client.list_tools()
    extension_added = "events_on_date" in {tool.name for tool in refreshed_tools}
    if extension_added:
        date_result = (await campus_client.call_tool("events_on_date", {"event_date": "2026-11-10"})).data
        print("Your MCP extension returned:", date_result)
    else:
        print("Extension not added yet. Add events_on_date above; this is your build task.")


## 9 Save your actual evidence

Record the catalogue you discovered, your changed query, a returned row and your new tool result. The evidence below distinguishes the working baseline from your unfinished extension.

In [ ]:
# Keep an honest record of the real protocol operations and the remaining task.
evidence = {"execution": "ACTUAL_MCP", "transport": "in-memory", "model_calls": 0,
    "baseline_event_ids": [row["event_id"] for row in events["events"]],
    "unknown_event_rejected": missing_result.data["ok"] is False,
    "resource_read": str(resources[0].uri), "prompt_retrieved": prompts[0].name,
    "save_confirmed": CONFIRM_SAVE, "new_tool_added_by_student": extension_added}
print(json.dumps(evidence, indent=2))
print("Core checks passed. Complete and explain the student tasks before claiming your project is finished.")


## Pair checkpoint and submission

Show your partner the client request, SQL parameters and actual result. Explain why a prompt template is not an executed LLM answer. Submit your saved notebook, one changed query, one new event, a successful `events_on_date` protocol call and a short explanation. Optional after class: connect a tool-capable model to these discovered MCP tools; the current core needs no model account.
